In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

import config
args = config.args

from helper import create_smart_stock_patterns, analyze_stock_mentions_fast, create_sequences

In [ ]:
#complete comment data from 2019 - 2024, from r/wallstreetbets; 
df_comments = pd.read_csv(args.submissionandcomments_dir)
df_comments

In [ ]:
#s&p stock information
df_stocks = pd.read_csv(args.sp_stocks_dir)

In [ ]:
#1. pair submission or comment with stock
stock_patterns = create_smart_stock_patterns(df_stocks)
mentions_df = analyze_stock_mentions_fast(df_comments, stock_patterns, batch_size=10000)
mentions_df['text_date'] = pd.to_datetime(mentions_df['created_utc'], unit='s', utc = True).dt.date
mentions_df['combined_text'] = (
    mentions_df['title'].fillna('') + ' ' + 
    mentions_df['selftext'].fillna('') + ' ' + 
    mentions_df['body'].fillna('')
).str.strip()

In [ ]:
#aggregate text by stock-date
results_df = mentions_df.groupby(['ticker', 'text_date']).agg({
    'combined_text': lambda x: ' '.join(x), 
    'id': 'count' ,
    'score': 'sum'
}).reset_index()

results_df['text_date'] = pd.to_datetime(results_df['text_date'])
results_df.columns = ['ticker', 'date', 'combined_text', 'num_posts', 'num_of_net_upvotes']

In [ ]:
#2. create sequence for traing
#earnings surprise information, including: fiscal end date, reported date, reported eps, estimated eps and surprise percent
earnings = pd.read_csv(args.eps_surprise_dir) 
earnings['reddit_sequence'] = earnings.apply(
    lambda row: create_sequences(row, results_df), axis=1
)

# earnings dataframe now has:
# ticker | earnings_date | surprise | reddit_sequence
# TSLA   | 2020-02-15    | +5%      | [day1_text, day2_text, ..., day60_text]
# AAPL   | 2020-02-20    | -2%      | [day1_text, day2_text, ..., day60_text]

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
model = AutoModel.from_pretrained('ProsusAI/finbert')

def get_bert_embedding(text):
    if not text or text.strip() == '':  # Handle empty days
        return torch.zeros(768)
    
    inputs = tokenizer(text, return_tensors='pt', 
                      truncation=True, max_length=512,
                      padding=True)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Use [CLS] token embedding
    embedding = outputs.last_hidden_state[:, 0, :].squeeze()
    return embedding

# Apply to each day in each sequence
def embed_sequence(text_sequence):
    embeddings = [get_bert_embedding(text) for text in text_sequence]
    return torch.stack(embeddings) 

earnings['embedding_sequence'] = earnings['reddit_sequence'].apply(embed_sequence)
# ticker | earnings_date | surprise | embedding_sequence
# TSLA   | 2020-02-15    | +5%      | Tensor(60, 768)
# AAPL   | 2020-02-20    | -2%      | Tensor(60, 768)

In [ ]:
earnings = earnings.sort_values('earnings_date').reset_index(drop=True)

# Define your date splits 
train_end_date = pd.to_datetime('2022-12-31')
val_end_date = pd.to_datetime('2023-12-31')
# Test starts after val_end_date and goes to end of data

# Create boolean masks based on dates
train_mask = earnings['earnings_date'] <= train_end_date
val_mask = (earnings['earnings_date'] > train_end_date) & (earnings['earnings_date'] <= val_end_date)
test_mask = earnings['earnings_date'] > val_end_date

# Split data using date-based masks
train_earnings = earnings[train_mask]
val_earnings = earnings[val_mask]
test_earnings = earnings[test_mask]

# Stack embeddings into tensors
X_train = torch.stack(train_earnings['embedding_sequence'].tolist())
y_train = torch.tensor((train_earnings['surprise'] > 0).values, dtype=torch.float32)

X_val = torch.stack(val_earnings['embedding_sequence'].tolist())
y_val = torch.tensor((val_earnings['surprise'] > 0).values, dtype=torch.float32)

X_test = torch.stack(test_earnings['embedding_sequence'].tolist())
y_test = torch.tensor((test_earnings['surprise'] > 0).values, dtype=torch.float32)

save_path = args.dataset_save_dir
dir = Path(save_path)
dir.mkdir(parents=True, exist_ok=True)

torch.save(X_train, Path(save_path) / 'X_train.pt')
torch.save(y_train, Path(save_path) / 'y_train.pt')
torch.save(X_val, Path(save_path) / 'X_val.pt')
torch.save(y_val, Path(save_path) / 'y_val.pt')
torch.save(X_test, Path(save_path) / 'X_testb.pt')
torch.save(y_test, Path(save_path) / 'y_test.pt')